# NeuroDecode–LaBraM | Kaggle full-data experiment (Phase 8)

**Goal:** Run **one complete, subject-held-out motor-imagery EEG training experiment** per Kaggle session with an immutable GitHub commit, a pinned LaBraM source, and reproducible evaluation artifacts. This is not a synthetic demonstration.

**Kaggle settings before `Run All`:**
1. **Accelerator → GPU** (a single GPU is sufficient; any second GPU is unused).
2. **Internet → On** to install packages and obtain public PhysioNet EEG and Hugging Face model weights. Without Internet, attach an already validated EEG cache and provide required packages/weights separately.
3. Use **Run All**. The first run will download public recordings and a pretrained model; the dataset preparation step can be substantial.
4. After completion, **Save Version**, preserve `/kaggle/working/neurodecode_outputs/`, and download `results_only_*.zip`. Never publish raw EEG recordings or a subject cache in the GitHub repository.

> **Important:** A successful Notebook is an experiment, not an accepted scientific paper. Compare identical, fixed subject splits over repeated model seeds. **No metrics are prefilled.** This Notebook has been structurally validated, but a full Kaggle GPU run has not yet been executed by ChatGPT.


In [ ]:
# ==========================================================
# EDIT ONLY THIS CELL BETWEEN EXPERIMENT RUNS
# ==========================================================
EXPERIMENT = "pretrained"           # "pretrained" or "scratch"
SEED = 42                           # Repeat for: 42, 123, 2025
SPLIT_SEED = 42                     # NEVER change between compared runs
BATCH_SIZE = 16                     # Reduce to 8 or 4 if GPU runs out of memory
MAX_EPOCHS = 30
EARLY_STOPPING_PATIENCE = 5
RUN_LABEL = ""                       # If retrying a failed run, e.g. "retry2"

# Optional: path to a previously validated cache you attached as a Kaggle Dataset.
# Example: "/kaggle/input/my-private-eeg-cache/physionet_mi_cache.npz"
ATTACHED_CACHE_PATH = ""

# The training code is pinned to this previously reviewed commit. Do not swap to
# GitHub main mid-campaign; doing so could invalidate cross-run comparisons.
REPOSITORY = "https://github.com/mohamad679/labram-mi-project.git"
SOURCE_COMMIT = "da4d2d3dfeb91fd553ec642240b8c39cd9d7a097"

assert EXPERIMENT in ("pretrained", "scratch")
assert SEED in (42, 123, 2025), "For the planned campaign, choose 42, 123, or 2025"
assert SPLIT_SEED == 42, "Fixed subject split is mandatory for comparability"
assert 1 <= BATCH_SIZE <= 64 and 1 <= MAX_EPOCHS <= 200
print(f"Selected run: {EXPERIMENT}, seed={SEED}, split_seed={SPLIT_SEED}, batch={BATCH_SIZE}")


## 1. GPU and storage preflight

Fails **before downloading the dataset** if Kaggle does not have an accessible NVIDIA GPU. The CUDA/PyTorch runtime is subsequently checked after installation.


In [ ]:
import os, sys, subprocess, shutil, json, time
from pathlib import Path

assert Path("/kaggle/working").exists(), "Run this notebook on Kaggle, not a standard local kernel."
try:
    info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                          text=True, capture_output=True, check=True)
except (FileNotFoundError, subprocess.CalledProcessError) as exc:
    raise RuntimeError("Kaggle GPU not available. Settings → Accelerator → GPU, then restart the session.") from exc
print("GPU(s):\n", info.stdout.strip())
free_gb = shutil.disk_usage("/kaggle/working").free / (1024**3)
print(f"Available /kaggle/working disk space: {free_gb:.1f} GiB")
if free_gb < 8:
    raise RuntimeError("Under 8 GiB of free disk. Clear old outputs or attach a validated cache before training.")
WORK = Path("/kaggle/working")
OUTPUT_ROOT = WORK / "neurodecode_outputs"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
RUN_DIR = OUTPUT_ROOT / f"{EXPERIMENT}_seed_{SEED}{'_' + RUN_LABEL if RUN_LABEL else ''}"
RUN_DIR.mkdir(parents=True, exist_ok=True)
print("Run output directory:", RUN_DIR)


## 2. Install a known compatible GPU environment

The pinned CPU integration test previously succeeded with PyTorch 2.5.1 and Braindecode 1.5.1. This cell installs the corresponding **CUDA 12.1** PyTorch wheels so the Kaggle GPU can be used. The dependency installation and full Kaggle training have not previously been guaranteed on every Kaggle image. If the package installation fails, keep the full error log for diagnosis rather than changing versions at random.

**Internet access is necessary for a first installation.**


In [ ]:
from pathlib import Path
import subprocess, sys

# Keep setup output in a file as well as the Kaggle notebook log.
install_log = RUN_DIR / "installation.log"
commands = [
  [sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "--no-input", 
   "--index-url", "https://download.pytorch.org/whl/cu121",
   "torch==2.5.1", "torchaudio==2.5.1", "torchvision==0.20.1"],
  [sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "--no-input",
   "braindecode[hub]==1.5.1", "moabb==1.7.2", "mne==1.13.2",
   "safetensors>=0.4,<1", "scikit-learn>=1.3,<2", "numpy>=1.26,<3",
   "PyYAML>=6,<7", "matplotlib>=3.8,<4", "pytest>=8,<9"],
]
with install_log.open("w", encoding="utf-8") as log:
    for cmd in commands:
        print("Installing:", " ".join(cmd[4:]), flush=True)
        p = subprocess.run(cmd, text=True, stdout=log, stderr=subprocess.STDOUT)
        if p.returncode:
            tail = install_log.read_text(encoding="utf-8", errors="replace")[-7000:]
            print(tail)
            raise RuntimeError(f"Dependency installation failed (exit {p.returncode}); see {install_log}")
print("Dependency installation succeeded. Full log:", install_log)


## 3. Clone immutable experiment source and verify GPU libraries

The pinned commit is intentionally **not** the moving `main` branch. This guarantees that the six planned runs use the same source code even if the GitHub repository evolves.


In [ ]:
import os, subprocess, sys, json
from pathlib import Path

SOURCE_DIR = WORK / "neurodecode_source"
if not (SOURCE_DIR / ".git").is_dir():
    subprocess.run(["git", "clone", REPOSITORY, str(SOURCE_DIR)], check=True)
subprocess.run(["git", "-C", str(SOURCE_DIR), "checkout", "--detach", SOURCE_COMMIT], check=True)
actual_sha = subprocess.check_output(["git", "-C", str(SOURCE_DIR), "rev-parse", "HEAD"], text=True).strip()
assert actual_sha == SOURCE_COMMIT
print("Reproducible source:", actual_sha)

probe = """import torch, torchaudio, braindecode, moabb, mne
assert torch.cuda.is_available(), "PyTorch cannot see CUDA. Restart Kaggle GPU session after installation."
assert torch.__version__.startswith("2.5.1"), torch.__version__
assert torchaudio.__version__.startswith("2.5.1"), torchaudio.__version__
print("torch", torch.__version__, "torchaudio", torchaudio.__version__, "braindecode", braindecode.__version__)
print("moabb", moabb.__version__, "mne", mne.__version__)
print("device", torch.cuda.get_device_name(0), "memory_GiB", round(torch.cuda.get_device_properties(0).total_memory/2**30,2))
"""
subprocess.run([sys.executable, "-c", probe], check=True, cwd=SOURCE_DIR)

# Fast CPU-only tests: these detect a broken installation before costly data downloads.
with (RUN_DIR / "unit_tests.log").open("w", encoding="utf-8") as log:
    result = subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=SOURCE_DIR,
                            text=True, stdout=log, stderr=subprocess.STDOUT)
print((RUN_DIR / "unit_tests.log").read_text(encoding="utf-8")[-3000:])
if result.returncode:
    raise RuntimeError("Repository offline tests failed. Do not begin GPU training.")


## 4. Reuse or build a verified 104-subject EEG cache

This project excludes five known problematic subject IDs from 109 public subjects. The source loader checks 64-channel order, 160 Hz, 400 time points, signal units (`µV`), and Cache Schema v2.

On a **fresh Kaggle session**, public-data downloads may take significant time and space. If you have a previously produced **validated cache**, attach it as a private Kaggle Dataset and set `ATTACHED_CACHE_PATH` above. The Cache file must never be uploaded to the public GitHub repository. The cache creation is done in a child process, allowing memory to be freed before training.


In [ ]:
import os, subprocess, sys
from pathlib import Path

# Optional: discover one attached validated cache by filename.
if ATTACHED_CACHE_PATH:
    cache = Path(ATTACHED_CACHE_PATH)
else:
    attached = list(Path("/kaggle/input").rglob("physionet_mi_cache.npz")) if Path("/kaggle/input").is_dir() else []
    if len(attached) > 1:
        raise RuntimeError("More than one attached EEG cache found; set ATTACHED_CACHE_PATH explicitly.")
    cache = attached[0] if attached else WORK / "physionet_mi_cache.npz"
if cache.is_dir():
    raise ValueError("Cache path points to a directory, not a .npz file")
print("EEG cache:", cache, "exists:", cache.is_file())
training_env = os.environ.copy()
training_env.update({
    "PHYSIONET_CACHE_PATH": str(cache),
    "PYTHONUNBUFFERED": "1",
    "MNE_DATA": str(WORK / "mne_data"),
    "MNE_DATASETS_EEGBCI_PATH": str(WORK / "mne_data"),
    "HF_HUB_DISABLE_TELEMETRY": "1",
})
Path(training_env["MNE_DATA"]).mkdir(parents=True, exist_ok=True)
# Do not enable ALLOW_UNVERIFIED_EEG_CACHE, even if an old cache is attached.
training_env.pop("ALLOW_UNVERIFIED_EEG_CACHE", None)

# Build/cache full public EEG data once (or validate attached cache).
# This subprocess releases its RAM before the training subprocess starts.
prepare_code = """from src.config import load_training_config
from src.train import load_dataset
from moabb.datasets import PhysionetMI
cfg = load_training_config('configs/baseline.yaml')
X, y, subjects = load_dataset(cfg)
expected_subjects = set(PhysionetMI().subject_list) - set(cfg.excluded_subjects)
observed = set(map(int, subjects))
if observed != expected_subjects:
    raise RuntimeError('Cache subject set does not match the full eligible PhysioNet cohort')
if X.shape[1:] != (64, 400):
    raise RuntimeError('EEG shape is incompatible with trained model')
print('FULL COHORT VERIFIED:', len(observed), 'subjects', len(y), 'EEG epochs', X.shape)
"""
log_path = RUN_DIR / "prepare_dataset.log"
with log_path.open("w", encoding="utf-8") as log:
    p = subprocess.Popen([sys.executable, "-u", "-c", prepare_code], cwd=SOURCE_DIR,
                         env=training_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
        log.write(line)
    status = p.wait()
if status:
    raise RuntimeError(f"Dataset preparation failed (exit {status}). See {log_path}.")
print("Full subject cohort checked; cache will be reused during training.")


## 5. Configure the single complete experiment

**One run = one model initialization + one seed.** All runs use the same `SPLIT_SEED=42`. This session will not automatically start the remaining seeds. The config and later logs are stored under a run-specific output folder and can be retained using Kaggle Save Version.


In [ ]:
import yaml, json, hashlib
from pathlib import Path

source_config = SOURCE_DIR / ("configs/pretrained_labram.yaml" if EXPERIMENT=="pretrained" else "configs/baseline.yaml")
cfg = yaml.safe_load(source_config.read_text(encoding="utf-8"))
cfg.update({
  "seed": SEED,
  "split_seed": SPLIT_SEED,
  "batch_size": BATCH_SIZE,
  "epochs": MAX_EPOCHS,
  "patience": EARLY_STOPPING_PATIENCE,
  "checkpoint_path": str(RUN_DIR / "model.pt"),
})
assert cfg["model_init"] == EXPERIMENT
assert cfg["split_seed"] == 42
if EXPERIMENT == "pretrained":
    assert cfg["pretrained_repo_id"] == "braindecode/labram-pretrained"
    assert cfg["pretrained_revision"] == "0563b6c626e7b40d9a36653b763715db94d945d7"
else:
    assert cfg["pretrained_repo_id"] is None and cfg["pretrained_revision"] is None

config_path = RUN_DIR / "experiment.yaml"
config_path.write_text(yaml.safe_dump(cfg, sort_keys=True), encoding="utf-8")
# Validate with exactly the repo implementation, not by guessing YAML structure.
sys.path.insert(0, str(SOURCE_DIR))
from src.config import load_training_config
validated = load_training_config(config_path)
print("Validated experiment:", validated.model_init, "seed", validated.seed, "split_seed", validated.split_seed)

meta_path = RUN_DIR / "model.json"
checkpoint = RUN_DIR / "model.pt"
if checkpoint.exists() and meta_path.exists():
    old = json.loads(meta_path.read_text(encoding="utf-8"))
    if old.get("status") == "finished" and old.get("configuration") == {**cfg, "class_names": cfg["class_names"], "excluded_subjects": cfg["excluded_subjects"]}:
        print("Previous matching completed run detected; training can be skipped safely.")
    else:
        raise RuntimeError("Existing checkpoint has no matching completed metadata; use a new clean run folder.")
elif checkpoint.exists():
    raise RuntimeError("Incomplete prior checkpoint found. Choose another run directory; no automatic resume is supported.")


## 6. Execute full-data EEG training

Training uses *all eligible subjects* with a subject-disjoint Train/Validation/Test split. The best checkpoint is selected on **validation Macro-F1**; test subjects are evaluated only after selection. Training output is streamed to `train.log`. On GPU out-of-memory, rerun from a clean run folder with smaller `BATCH_SIZE` (8 or 4).


In [ ]:
import os, subprocess, sys, json
from pathlib import Path

skip_completed = False
if checkpoint.is_file() and meta_path.is_file():
    old = json.loads(meta_path.read_text(encoding="utf-8"))
    skip_completed = (old.get("status") == "finished")

if skip_completed:
    print("Using existing completed run:", RUN_DIR)
else:
    cmd = [sys.executable, "-u", "-m", "src.train", "--config", str(config_path)]
    print("Running:", " ".join(cmd))
    log_path = RUN_DIR / "train.log"
    with log_path.open("w", encoding="utf-8") as log:
        p = subprocess.Popen(cmd, cwd=SOURCE_DIR, env=training_env,
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             text=True, bufsize=1)
        for line in p.stdout:
            print(line, end="")
            log.write(line)
        returncode = p.wait()
    if returncode != 0:
        print("TRAINING FAILED. Last 5000 log characters:")
        print(log_path.read_text(encoding="utf-8", errors="replace")[-5000:])
        raise RuntimeError(f"Real-data training failed with exit {returncode}; see {log_path}")

prediction_file = RUN_DIR / "heldout_predictions.npz"
assert checkpoint.is_file() and meta_path.is_file() and prediction_file.is_file(), "Expected training artifacts missing."
metadata = json.loads(meta_path.read_text(encoding="utf-8"))
assert metadata.get("status") == "finished"
assert metadata["configuration"]["seed"] == SEED and metadata["configuration"]["split_seed"] == SPLIT_SEED
train_sub = set(metadata["train_subjects"])
val_sub = set(metadata["validation_subjects"])
test_sub = set(metadata["test_subjects"])
assert not (train_sub & val_sub or train_sub & test_sub or val_sub & test_sub), "Subject overlap detected."
assert (len(train_sub), len(val_sub), len(test_sub)) == (75,14,15), "Unexpected cohort split size."
print("Full training complete. Test accuracy:", metadata["test_accuracy"],
      "Macro-F1:", metadata["test_macro_f1"])
print("Checkpoint SHA256:", metadata["checkpoint_sha256"])


## 7. Generate genuine scientific figures and participant-aware intervals

All results below are computed from the **saved held-out probabilities**; nothing is manually inserted or simulated. Confidence intervals resample **participants**, not correlated within-participant EEG epochs.


In [ ]:
import subprocess, sys, json
from pathlib import Path

EVAL_DIR = RUN_DIR / "evaluation"
EVAL_DIR.mkdir(parents=True, exist_ok=True)
commands = [
  [sys.executable, "-m", "scripts.build_benchmark", "--predictions", str(prediction_file),
   "--output-dir", str(EVAL_DIR)],
  [sys.executable, "-m", "scripts.subject_confidence", "--predictions", str(prediction_file),
   "--output", str(EVAL_DIR / "subject_ci.json"), "--n-bootstraps", "1000", "--seed", "42"],
]
for cmd in commands:
    subprocess.run(cmd, cwd=SOURCE_DIR, env=training_env, check=True)
# Extract actual epoch history from this run's training log for learning curves.
import re, csv
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
training_log = (RUN_DIR / "train.log").read_text(encoding="utf-8")
pattern = re.compile(
    r"Epoch\s+(\d+)\s*\|\s*train_loss=([0-9.eE+-]+)\s+acc=([0-9.eE+-]+)\s+f1=([0-9.eE+-]+)"
    r"\s*\|\s*val_loss=([0-9.eE+-]+)\s+acc=([0-9.eE+-]+)\s+f1=([0-9.eE+-]+)"
)
history = [tuple(float(x) for x in row) for row in pattern.findall(training_log)]
if not history:
    raise RuntimeError("No parseable per-epoch logs found; cannot create an evidence-backed learning curve.")
with (EVAL_DIR / "training_history.csv").open("w", encoding="utf-8", newline="") as handle:
    writer = csv.writer(handle)
    writer.writerow(["epoch", "train_loss", "train_accuracy", "train_macro_f1", "val_loss", "val_accuracy", "val_macro_f1"])
    writer.writerows(history)
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot([r[0] for r in history], [r[3] for r in history], marker="o", label="Training Macro-F1")
ax.plot([r[0] for r in history], [r[6] for r in history], marker="o", label="Validation Macro-F1")
ax.set(xlabel="Epoch", ylabel="Macro-F1", title="Actual training/validation history")
ax.legend()
fig.tight_layout()
fig.savefig(EVAL_DIR / "learning_curves.png", dpi=180)
plt.close(fig)
metrics = json.loads((EVAL_DIR / "metrics.json").read_text(encoding="utf-8"))
ci = json.loads((EVAL_DIR / "subject_ci.json").read_text(encoding="utf-8"))
assert metrics["n_test_subjects"] == 15 and ci["n_heldout_subjects"] == 15
print("Validated held-out metrics:", {k:metrics[k] for k in ("accuracy","macro_f1","balanced_accuracy","ece_10_bins")})
print("Subject-cluster 95% CI:", json.dumps({"accuracy":ci["accuracy"], "macro_f1":ci["macro_f1"]}, indent=2))
from IPython.display import display, Image
for fig in ["confusion_matrix.png", "reliability_diagram.png", "learning_curves.png"]:
    display(Image(filename=str(EVAL_DIR / fig), width=560))


## 8. Archive verifiable evidence (excluding EEG and weights)

You can download the **small, shareable results ZIP** and send it here for independent analysis. It intentionally excludes EEG cache and model weights. The full model checkpoint is available separately in Kaggle Outputs. Save Version is recommended so the run survives after the session ends.

Never add this Notebook's caches, patient-level predictions or model checkpoints to a public GitHub commit.


In [ ]:
import json, hashlib, os, subprocess, sys, zipfile
from pathlib import Path

freeze_file = RUN_DIR / "pip-freeze.txt"
freeze_file.write_text(subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True), encoding="utf-8")

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

manifest = {
    "project": "NeuroDecode-LaBraM",
    "source_commit": actual_sha,
    "model_init": EXPERIMENT,
    "model_seed": SEED,
    "split_seed": SPLIT_SEED,
    "model_checkpoint_sha256": sha256_file(checkpoint),
    "prediction_file_sha256": sha256_file(prediction_file),
    "full_cohort_subjects": len(set(metadata["train_subjects"] + metadata["validation_subjects"] + metadata["test_subjects"])),
    "test_subjects": len(metadata["test_subjects"]),
    "accuracy": metrics["accuracy"],
    "macro_f1": metrics["macro_f1"],
    "bootstrap_method": ci["method"],
    "cache_path": str(cache),
    "cache_sha256": None,  # No misleading provenance claim; large cache may be separately hashed.
    "checkpoint_is_distributed": False,
    "results_are_full_training": True,
}
manifest_file = RUN_DIR / "run-manifest.json"
manifest_file.write_text(json.dumps(manifest, indent=2, ensure_ascii=False)+"\n", encoding="utf-8")

bundle = OUTPUT_ROOT / f"results_only_{EXPERIMENT}_seed_{SEED}{'_' + RUN_LABEL if RUN_LABEL else ''}.zip"
allowed = [config_path, meta_path, manifest_file, freeze_file, RUN_DIR/"train.log", RUN_DIR/"unit_tests.log",
           RUN_DIR/"prepare_dataset.log", RUN_DIR/"installation.log"]
allowed += sorted(EVAL_DIR.glob("*"))
with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    for file in allowed:
        if file.is_file():
            archive.write(file, arcname=str(file.relative_to(RUN_DIR)))
print("\nREADY. Download/share this result ZIP (without raw EEG, weight files or subject prediction NPZ):", bundle)
print("Saved separate model checkpoint:", checkpoint)
print("Saved validated local cache:", cache)
print("To complete the controlled study, repeat with:")
for model in ["pretrained", "scratch"]:
    for seed in [42, 123, 2025]:
        print(" -", model, seed)
print("Keep SPLIT_SEED=42 and the same SOURCE_COMMIT in every run.")


## After the first run: what to do next

1. Download the `results_only_pretrained_seed_42.zip` (or corresponding model/seed name) from Kaggle Outputs and send it back for review.
2. **Save Version** and preserve the validated cache as a **private** Kaggle Dataset or reuse a permitted attached cache in subsequent sessions. If `ATTACHED_CACHE_PATH` is set, data will not be re-downloaded; keep Kaggle notebook/output visibility **private** while the cache is present.
3. Repeat one run at a time for the remaining combinations: `pretrained × (42, 123, 2025)` and `scratch × (42, 123, 2025)`. Keep `SPLIT_SEED=42` and `SOURCE_COMMIT` fixed. It is acceptable to run on different GPU sessions if the rest of the protocol stays identical.
4. **Important limitation:** Resuming partway through a failed epoch is **not yet implemented** in `src.train`. A failed run must restart from a clean run-specific output directory. Keep logs for troubleshooting.
5. For Kaggle GPU out-of-memory, try `BATCH_SIZE=8` or `4`. Record any changes; a fair comparison uses the same effective training protocol for both conditions.
6. Full multi-seed comparison, any claims of improvement, and publication-grade reporting require all six successful runs. A first successful run is only one empirical observation.

**Research ethics:** The dataset is publicly available but must be used according to its own license. Do not distribute subject-level recordings or caches by accident. EEG model predictions are research results, not clinical diagnosis or a substitute for medical assessment.
